# ML Assignment 2 - Complete Setup Notebook

**Run all cells top to bottom.**

This notebook will:
1. Install all packages
2. Train 5 ML models (Breast Cancer dataset)
3. Save model files to `model/`
4. Save `test_data.csv`
5. Write `app.py`, `requirements.txt`, `README.md`
6. Launch the Streamlit app

---

## Cell 1 - Install Packages

In [3]:
import sys, subprocess

# Upgrade to numpy-compatible versions of scipy/sklearn for Python 3.12
packages = [
    "numpy",
    "scipy",
    "scikit-learn",
    "pandas",
    "matplotlib",
    "seaborn",
    "joblib",
    "streamlit",
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade", pkg, "-q"],
        capture_output=True, text=True
    )
    print(f"  installed/upgraded: {pkg}")

print()
print("IMPORTANT: Now restart the kernel before running Cell 2!")
print("Go to: Kernel menu -> Restart Kernel (or press 0, 0)")
print("After restart, run Cell 2 onwards (skip Cell 1).")

  installed/upgraded: numpy
  installed/upgraded: scipy
  installed/upgraded: scikit-learn
  installed/upgraded: pandas
  installed/upgraded: matplotlib
  installed/upgraded: seaborn
  installed/upgraded: joblib
  installed/upgraded: streamlit

IMPORTANT: Now restart the kernel before running Cell 2!
Go to: Kernel menu -> Restart Kernel (or press 0, 0)
After restart, run Cell 2 onwards (skip Cell 1).


## Cell 2 - Train All 5 Models and Save Files

In [5]:
import os, numpy as np, pandas as pd, joblib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

BASE_DIR  = os.getcwd()
MODEL_DIR = os.path.join(BASE_DIR, 'model')
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Working dir: {BASE_DIR}')

data = load_breast_cancer()
df   = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
print(f'Dataset: {df.shape}  Malignant(0)={(df.target==0).sum()}  Benign(1)={(df.target==1).sum()}')
df.to_csv(os.path.join(BASE_DIR, 'breast_cancer_full.csv'), index=False)

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.joblib'))

test_df = pd.DataFrame(X_test_s, columns=X.columns)
test_df['target'] = y_test.values
test_df.to_csv(os.path.join(BASE_DIR, 'test_data.csv'), index=False)
print(f'test_data.csv saved ({test_df.shape[0]} rows)')

models_dict = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':         GaussianNB(),
    'SVM':                 SVC(probability=True, random_state=42),
}
file_names = {
    'Logistic Regression': 'Logistic_Regression.joblib',
    'Decision Tree':       'Decision_Tree.joblib',
    'KNN':                 'KNN.joblib',
    'Naive Bayes':         'Naive_Bayes.joblib',
    'SVM':                 'SVM.joblib',
}

print(f"{'Model':<22} {'Acc':>6} {'AUC':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'MCC':>6}")
print('-'*60)
results = []
for name, model in models_dict.items():
    model.fit(X_train_s, y_train)
    yp = model.predict(X_test_s)
    yb = model.predict_proba(X_test_s)[:, 1]
    acc  = accuracy_score(y_test, yp)
    auc  = roc_auc_score(y_test, yb)
    prec = precision_score(y_test, yp)
    rec  = recall_score(y_test, yp)
    f1   = f1_score(y_test, yp)
    mcc  = matthews_corrcoef(y_test, yp)
    print(f'{name:<22} {acc:>6.4f} {auc:>6.4f} {prec:>6.4f} {rec:>6.4f} {f1:>6.4f} {mcc:>6.4f}')
    joblib.dump(model, os.path.join(MODEL_DIR, file_names[name]))
    results.append({'Model':name,'Accuracy':round(acc,4),'AUC':round(auc,4),
                    'Precision':round(prec,4),'Recall':round(rec,4),'F1':round(f1,4),'MCC':round(mcc,4)})
print('All models saved to model/ folder.')

Working dir: /home/s2026aj05521/workspace/ML_Assignment
Dataset: (569, 31)  Malignant(0)=212  Benign(1)=357
test_data.csv saved (114 rows)
Model                     Acc    AUC   Prec    Rec     F1    MCC
------------------------------------------------------------
Logistic Regression    0.9825 0.9954 0.9861 0.9861 0.9861 0.9623
Decision Tree          0.9123 0.9157 0.9559 0.9028 0.9286 0.8174
KNN                    0.9561 0.9788 0.9589 0.9722 0.9655 0.9054
Naive Bayes            0.9298 0.9868 0.9444 0.9444 0.9444 0.8492
SVM                    0.9825 0.9950 0.9861 0.9861 0.9861 0.9623
All models saved to model/ folder.


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## Cell 3 - Write app.py

In [7]:
app_content = "import streamlit as st\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport joblib\nimport os\nfrom sklearn.metrics import (\n    accuracy_score, roc_auc_score, precision_score,\n    recall_score, f1_score, matthews_corrcoef,\n    confusion_matrix, classification_report)\n\nst.set_page_config(page_title='Breast Cancer Classifier', page_icon='🔬', layout='wide')\n\nMODEL_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'model')\nMODEL_FILES = {\n    'Logistic Regression': 'Logistic_Regression.joblib',\n    'Decision Tree':       'Decision_Tree.joblib',\n    'KNN':                 'KNN.joblib',\n    'Naive Bayes':         'Naive_Bayes.joblib',\n    'SVM':                 'SVM.joblib',\n}\n\n@st.cache_resource\ndef load_model(name):\n    return joblib.load(os.path.join(MODEL_DIR, MODEL_FILES[name]))\n\ndef get_metrics(y_true, y_pred, y_prob):\n    return {\n        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),\n        'AUC':       round(roc_auc_score(y_true, y_prob), 4),\n        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),\n        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),\n        'F1 Score':  round(f1_score(y_true, y_pred, zero_division=0), 4),\n        'MCC':       round(matthews_corrcoef(y_true, y_pred), 4),\n    }\n\nst.title('Breast Cancer Classification - ML Assignment 2')\nst.markdown('Upload **test_data.csv**, select a model, and view evaluation metrics.')\nst.divider()\n\nwith st.sidebar:\n    st.header('Controls')\n    uploaded = st.file_uploader('Upload test_data.csv', type=['csv'])\n    model_choice = st.selectbox('Select Model', list(MODEL_FILES.keys()))\n    run_btn = st.button('Run Evaluation', type='primary', use_container_width=True)\n    st.divider()\n    st.markdown('**Dataset:** Wisconsin Breast Cancer | **Features:** 30 | **Instances:** 569')\n\nif uploaded is None:\n    st.info('Upload test_data.csv from the sidebar to get started.')\n    st.stop()\n\ntry:\n    df = pd.read_csv(uploaded)\nexcept Exception as e:\n    st.error(f'Could not read file: {e}')\n    st.stop()\n\nif 'target' not in df.columns:\n    st.error('CSV must contain a target column.')\n    st.stop()\n\nX = df.drop('target', axis=1).values\ny = df['target']\nst.success(f'Loaded {len(df)} samples with {X.shape[1]} features.')\n\nif not run_btn:\n    st.info('Click Run Evaluation in the sidebar.')\n    st.stop()\n\nwith st.spinner(f'Evaluating {model_choice}...'):\n    model  = load_model(model_choice)\n    y_pred = model.predict(X)\n    y_prob = model.predict_proba(X)[:, 1]\n    metrics = get_metrics(y, y_pred, y_prob)\n    cm = confusion_matrix(y, y_pred)\n    cr = classification_report(y, y_pred, target_names=['Malignant (0)', 'Benign (1)'])\n\nst.subheader(f'Evaluation Metrics - {model_choice}')\ncols = st.columns(6)\nfor col, (name, val) in zip(cols, metrics.items()):\n    col.metric(name, val)\n\nst.divider()\nc1, c2 = st.columns(2)\n\nwith c1:\n    st.subheader('Confusion Matrix')\n    fig, ax = plt.subplots(figsize=(5, 4))\n    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',\n                xticklabels=['Malignant', 'Benign'],\n                yticklabels=['Malignant', 'Benign'],\n                ax=ax, linewidths=0.5)\n    ax.set_xlabel('Predicted')\n    ax.set_ylabel('Actual')\n    ax.set_title(f'{model_choice}')\n    plt.tight_layout()\n    st.pyplot(fig)\n\nwith c2:\n    st.subheader('Classification Report')\n    st.code(cr)\n\nst.divider()\nst.subheader('All Models Comparison')\nall_res = []\nfor mname in MODEL_FILES:\n    m  = load_model(mname)\n    yp = m.predict(X)\n    yb = m.predict_proba(X)[:, 1]\n    mt = get_metrics(y, yp, yb)\n    all_res.append({'Model': mname, **mt})\nres_df = pd.DataFrame(all_res).set_index('Model')\nst.dataframe(res_df.style.highlight_max(axis=0, color='#d4edda'), use_container_width=True)\nst.caption('Green = best value per metric. Evaluated on uploaded test data without retraining.')\n"
with open(os.path.join(BASE_DIR, 'app.py'), 'w', encoding='utf-8') as f:
    f.write(app_content)
print('app.py written.')

app.py written.


## Cell 4 - Write requirements.txt

In [9]:
req = 'streamlit\nscikit-learn\nnumpy\npandas\nmatplotlib\nseaborn\njoblib\n'
with open(os.path.join(BASE_DIR, 'requirements.txt'), 'w') as f:
    f.write(req)
print('requirements.txt written.')

requirements.txt written.


## Cell 5 - Write README.md

In [11]:
readme_content = '# ML Assignment 2 - Breast Cancer Classification\n\n## a. Problem Statement\nBinary classification to predict whether a breast tumour is malignant (0) or benign (1) based on 30 numeric features from digitised images of FNA of breast masses. Five ML classifiers are trained, evaluated, and served through a Streamlit web application.\n\n---\n\n## b. Dataset Description\n| Property | Value |\n|---|---|\n| Name | Wisconsin Breast Cancer Dataset |\n| Source | sklearn.datasets / UCI ML Repository |\n| Instances | 569 |\n| Features | 30 numeric |\n| Target | Binary: 0 Malignant (212), 1 Benign (357) |\n| Train/Test Split | 80/20 stratified, random_state=42 |\n| Preprocessing | StandardScaler |\n\n---\n\n## c. GitHub Repository Link\nhttps://github.com/YOUR_USERNAME/ml-assignment-2\n\n---\n\n## d. Models and Evaluation Metrics\n\n### Comparison Table\n| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |\n|---|---|---|---|---|---|---|\n| Logistic Regression | 0.9825 | 0.9954 | 0.9861 | 0.9861 | 0.9861 | 0.9623 |\n| Decision Tree | 0.9123 | 0.9157 | 0.9559 | 0.9028 | 0.9286 | 0.8174 |\n| KNN | 0.9561 | 0.9788 | 0.9589 | 0.9722 | 0.9655 | 0.9054 |\n| Naive Bayes | 0.9298 | 0.9868 | 0.9444 | 0.9444 | 0.9444 | 0.8492 |\n| Support Vector Machine | 0.9825 | 0.9950 | 0.9861 | 0.9861 | 0.9861 | 0.9623 |\n\n### Observations\n| ML Model Name | Observation |\n|---|---|\n| Logistic Regression | Best overall - highest accuracy 98.25% and AUC 0.9954. Linear boundary works well. Very low false-negative rate makes it clinically reliable. |\n| Decision Tree | Lowest performer (91.23% accuracy). Axis-aligned boundaries overfit training data. Sensitive to individual feature thresholds. |\n| KNN | Good performance (95.61% accuracy). High recall means fewer missed malignant cases. Depends on feature scaling and k. |\n| Naive Bayes | Moderate accuracy (92.98%) but high AUC (0.9868). Independence assumption violated by correlated features. |\n| Support Vector Machine | Tied with LR on all hard-threshold metrics. RBF kernel finds near-perfect margin in high-dimensional space. |\n\n### Overall Winner\nLogistic Regression - best AUC and tied best on all other metrics. Highly interpretable and clinically reliable.\n\n---\n\n## Streamlit App\nhttps://YOUR_APP_NAME.streamlit.app\n'
with open(os.path.join(BASE_DIR, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(readme_content)
print('README.md written.')

README.md written.


## Cell 6 - Verify All Files Created

In [13]:
expected = [
    'app.py','requirements.txt','README.md','test_data.csv','breast_cancer_full.csv',
    os.path.join('model','scaler.joblib'),
    os.path.join('model','Logistic_Regression.joblib'),
    os.path.join('model','Decision_Tree.joblib'),
    os.path.join('model','KNN.joblib'),
    os.path.join('model','Naive_Bayes.joblib'),
    os.path.join('model','SVM.joblib'),
]
print('Files in:', BASE_DIR)
all_ok = True
for fname in expected:
    full   = os.path.join(BASE_DIR, fname)
    exists = os.path.exists(full)
    size   = os.path.getsize(full) if exists else 0
    print(f"  [{'OK' if exists else 'MISSING'}]  {fname}  ({size} bytes)")
    if not exists: all_ok = False
print()
print('ALL FILES CREATED!' if all_ok else 'Some files missing - re-run cells above.')

Files in: /home/s2026aj05521/workspace/ML_Assignment
  [OK]  app.py  (3907 bytes)
  [OK]  requirements.txt  (62 bytes)
  [OK]  README.md  (2244 bytes)
  [OK]  test_data.csv  (68319 bytes)
  [OK]  breast_cancer_full.csv  (121385 bytes)
  [OK]  model/scaler.joblib  (2103 bytes)
  [OK]  model/Logistic_Regression.joblib  (1103 bytes)
  [OK]  model/Decision_Tree.joblib  (4265 bytes)
  [OK]  model/KNN.joblib  (113652 bytes)
  [OK]  model/Naive_Bayes.joblib  (1735 bytes)
  [OK]  model/SVM.joblib  (26907 bytes)

ALL FILES CREATED!


## Cell 7 - Metrics Summary Table

In [15]:
import pandas as pd
summary = pd.DataFrame(results)
print('EVALUATION METRICS ON TEST SET')
print('='*65)
print(summary.to_string(index=False))
print('='*65)
print('Winner: Logistic Regression (highest AUC = 0.9954)')

EVALUATION METRICS ON TEST SET
              Model  Accuracy    AUC  Precision  Recall     F1    MCC
Logistic Regression    0.9825 0.9954     0.9861  0.9861 0.9861 0.9623
      Decision Tree    0.9123 0.9157     0.9559  0.9028 0.9286 0.8174
                KNN    0.9561 0.9788     0.9589  0.9722 0.9655 0.9054
        Naive Bayes    0.9298 0.9868     0.9444  0.9444 0.9444 0.8492
                SVM    0.9825 0.9950     0.9861  0.9861 0.9861 0.9623
Winner: Logistic Regression (highest AUC = 0.9954)


## Cell 8 - Launch Streamlit App

Runs the app in the background. Open **http://localhost:8501** in your browser.

Steps:
1. Upload `test_data.csv` in the sidebar
2. Select any model from the dropdown
3. Click **Run Evaluation**
4. **TAKE A SCREENSHOT** for your 1-mark lab proof

In [17]:
import subprocess, sys, time
app_path = os.path.join(BASE_DIR, 'app.py')
proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', app_path, '--server.headless', 'true'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)
print('Streamlit app is running!')
print('Open in browser: http://localhost:8501')
print()
print('1. Upload test_data.csv')
print('2. Select model from dropdown')
print('3. Click Run Evaluation')
print('4. TAKE A SCREENSHOT for lab proof')

Streamlit app is running!
Open in browser: http://localhost:8501

1. Upload test_data.csv
2. Select model from dropdown
3. Click Run Evaluation
4. TAKE A SCREENSHOT for lab proof
